# Definición del problema y del proyecto — Fase 1

**Proyecto:** Análisis de licitaciones públicas en Chile, enero de 2025  
**Grupo:** 2 · MCDI-500 · Programación para la Ciencia de Datos  
**Integrantes:** Ernesto Siebert Aguilera · Oscar Salgado Sanhueza · Bertgil Gonzalez Olmos · Daniel Rojas Sánchez

Este cuaderno define el problema, los objetivos y las preguntas del proyecto; verifica el entorno y la estructura del repositorio; justifica el conjunto de datos de ChileCompra y comprueba que los indicadores propuestos pueden calcularse con las variables disponibles. El archivo `data/raw/04LicitacionesEne.csv` solo se lee: la limpieza y las conversiones definitivas se realizan en la Fase 2.

## Introducción

Las instituciones públicas chilenas compran bienes y servicios mediante licitaciones con múltiples oferentes, rubros y monedas. ChileCompra publica estos registros como datos abiertos, lo que permite describir cómo compiten los proveedores, cómo se distribuyen las adjudicaciones y dónde aparecen comportamientos poco habituales.

El proyecto combina dos miradas: una **descriptiva** (cómo se distribuyen ofertas, resultados y montos) y otra de **señales de alerta** (baja competencia, montos atípicos, plazos inusuales, concentración y posibles proveedores relacionados). Una señal de alerta indica un caso que merece revisión, no una irregularidad comprobada.

**Entregables de la fase:** informe técnico, repositorio con estructura y *commits* documentados, `README` técnico, este cuaderno ejecutado y el mapa conceptual de la Formativa 1.

In [1]:
import importlib
import platform
import shutil
import subprocess
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd

## 1. Problema, objetivos y preguntas

La definición queda en un diccionario del que se derivan la presentación, la validación y la trazabilidad. Cada objetivo específico tiene una pregunta de análisis asociada con el mismo número.

In [2]:
PROYECTO = {
    "titulo": "Análisis de licitaciones públicas en Chile, enero de 2025",
    "grupo": "Grupo 2",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": [
        "Ernesto Siebert Aguilera",
        "Oscar Salgado Sanhueza",
        "Bertgil Gonzalez Olmos",
        "Daniel Rojas Sánchez",
    ],
    "problematica": (
        "Las instituciones públicas compran bienes y servicios mediante licitaciones con "
        "múltiples oferentes y rubros. Se requiere describir cómo se distribuyen las ofertas "
        "y adjudicaciones e identificar señales de alerta —baja competencia, montos atípicos, "
        "plazos inusuales, concentración y posibles proveedores relacionados— que permitan "
        "priorizar procesos para una revisión más detallada."
    ),
    "objetivo_general": (
        "Describir las ofertas y adjudicaciones registradas por ChileCompra durante enero de "
        "2025 e identificar señales de alerta en competencia, montos, plazos y concentración "
        "de proveedores, sin atribuir causalidad ni irregularidades."
    ),
    "objetivos_especificos": [
        "Caracterizar la distribución de ofertas y sus resultados por tipo de licitación, "
        "rubro e institución.",
        "Medir el nivel de competencia e identificar licitaciones e ítems con un solo oferente.",
        "Detectar montos atípicos respecto de ofertas comparables (mismo ítem y moneda) y "
        "brechas relevantes entre la oferta adjudicada y las demás ofertas del mismo ítem.",
        "Identificar plazos inusualmente cortos entre publicación y cierre según el tipo de licitación.",
        "Describir la concentración de adjudicaciones por proveedor y los patrones repetitivos "
        "proveedor–organismo comprador.",
        "Explorar posibles proveedores relacionados: RUT distintos con nombres, actividad u "
        "ofertas muy similares que compiten en las mismas licitaciones o ganan en los mismos organismos.",
    ],
    "preguntas": [
        "¿Cómo se distribuyen las ofertas y sus resultados entre tipos de licitación, rubros e instituciones?",
        "¿Qué proporción de licitaciones e ítems recibe una sola oferta y en qué rubros u organismos se concentran?",
        "¿Qué ofertas se alejan de los montos de ítems comparables en la misma moneda y qué tan grande es "
        "la diferencia entre la oferta ganadora y las demás?",
        "¿Qué procesos tienen plazos entre publicación y cierre muy cortos en comparación con otros de su mismo tipo?",
        "¿Qué proporción de las adjudicaciones concentra cada proveedor y qué proveedores ganan "
        "reiteradamente en un mismo organismo?",
        "¿Existen proveedores ganadores con RUT distinto pero nombre, actividad o texto de oferta casi "
        "idénticos que coinciden en las mismas licitaciones u organismos?",
    ],
    "criterios_exito": [
        "La unidad de análisis y la definición de oferta ganadora quedan explícitas.",
        "Cada señal de alerta tiene una regla reproducible y un umbral justificado.",
        "Los montos se comparan solo dentro de la misma moneda y entre ítems comparables.",
        "Cada pregunta se vincula con variables concretas, una fase y una evidencia en el repositorio.",
        "Los hallazgos se presentan como señales para revisión, no como pruebas de irregularidad.",
    ],
    "alcance": {
        "incluye": [
            "Registros del archivo de ChileCompra correspondiente a enero de 2025.",
            "Unidad de análisis: una oferta de un proveedor para un ítem dentro de una licitación.",
            "Análisis descriptivo y construcción de indicadores de alerta a partir de los registros.",
        ],
        "excluye": [
            "Concluir fraude, colusión o responsabilidad de personas o empresas.",
            "Determinar vínculos legales entre empresas (socios, representantes o domicilios).",
            "Generalizar a otros meses o al año completo.",
            "Convertir monedas sin una fecha de referencia justificada.",
        ],
    },
    "limitaciones": [
        "Un solo mes no permite generalizar tendencias al año completo ni a otros periodos.",
        "Las filas representan ofertas de ítems, no licitaciones independientes.",
        "CLP, CLF, USD y UTM no son comparables sin una conversión y fecha de referencia.",
        "El archivo no incluye socios, representantes legales ni domicilios: la similitud entre "
        "proveedores es solo un indicio y su confirmación requiere fuentes externas.",
        "Un único oferente o un plazo corto pueden tener causas legítimas (rubros especializados, urgencias).",
        "Algunos proveedores aparecen anonimizados (por ejemplo, «PERSONA NATURAL»), lo que limita "
        "la comparación por nombre.",
    ],
}

In [3]:
CLAVES_OBLIGATORIAS = (
    "titulo", "problematica", "objetivo_general", "objetivos_especificos",
    "preguntas", "criterios_exito", "alcance", "limitaciones",
)


def presentar_proyecto(config):
    '''
    Imprime la definición del proyecto de forma legible.

    Parámetros
    ----------
    config : dict
        Diccionario de configuración del proyecto.

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    ValueError
        Si la cantidad de objetivos específicos y de preguntas no coincide.
    '''
    faltan = [c for c in CLAVES_OBLIGATORIAS if c not in config]
    if faltan:
        raise KeyError(f"Faltan claves obligatorias en la configuración: {faltan}")
    if len(config["objetivos_especificos"]) != len(config["preguntas"]):
        raise ValueError("Cada objetivo específico debe tener una pregunta asociada.")

    print(f"{config['titulo']} — {config['grupo']}")
    print("\nPROBLEMÁTICA\n ", config["problematica"])
    print("\nOBJETIVO GENERAL\n ", config["objetivo_general"])
    print("\nOBJETIVOS ESPECÍFICOS Y PREGUNTAS")
    for n, (obj, preg) in enumerate(zip(config["objetivos_especificos"], config["preguntas"]), start=1):
        print(f"  {n}. {obj}\n     → {preg}")
    print("\nCRITERIOS DE ÉXITO")
    for n, texto in enumerate(config["criterios_exito"], start=1):
        print(f"  {n}. {texto}")


presentar_proyecto(PROYECTO)

Análisis de licitaciones públicas en Chile, enero de 2025 — Grupo 2

PROBLEMÁTICA
  Las instituciones públicas compran bienes y servicios mediante licitaciones con múltiples oferentes y rubros. Se requiere describir cómo se distribuyen las ofertas y adjudicaciones e identificar señales de alerta —baja competencia, montos atípicos, plazos inusuales, concentración y posibles proveedores relacionados— que permitan priorizar procesos para una revisión más detallada.

OBJETIVO GENERAL
  Describir las ofertas y adjudicaciones registradas por ChileCompra durante enero de 2025 e identificar señales de alerta en competencia, montos, plazos y concentración de proveedores, sin atribuir causalidad ni irregularidades.

OBJETIVOS ESPECÍFICOS Y PREGUNTAS
  1. Caracterizar la distribución de ofertas y sus resultados por tipo de licitación, rubro e institución.
     → ¿Cómo se distribuyen las ofertas y sus resultados entre tipos de licitación, rubros e instituciones?
  2. Medir el nivel de competencia 

In [4]:
print("Alcance")
print("  Incluye:")
for item in PROYECTO["alcance"]["incluye"]:
    print(f"    · {item}")
print("  Excluye:")
for item in PROYECTO["alcance"]["excluye"]:
    print(f"    · {item}")

print("\nLimitaciones")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

Alcance
  Incluye:
    · Registros del archivo de ChileCompra correspondiente a enero de 2025.
    · Unidad de análisis: una oferta de un proveedor para un ítem dentro de una licitación.
    · Análisis descriptivo y construcción de indicadores de alerta a partir de los registros.
  Excluye:
    · Concluir fraude, colusión o responsabilidad de personas o empresas.
    · Determinar vínculos legales entre empresas (socios, representantes o domicilios).
    · Generalizar a otros meses o al año completo.
    · Convertir monedas sin una fecha de referencia justificada.

Limitaciones
  · Un solo mes no permite generalizar tendencias al año completo ni a otros periodos.
  · Las filas representan ofertas de ítems, no licitaciones independientes.
  · CLP, CLF, USD y UTM no son comparables sin una conversión y fecha de referencia.
  · El archivo no incluye socios, representantes legales ni domicilios: la similitud entre proveedores es solo un indicio y su confirmación requiere fuentes externas.
 

### 1.1 Señales de alerta propuestas

| Señal | Regla preliminar | Variables | Objetivo |
| --- | --- | --- | --- |
| Baja competencia | Licitación o ítem con un solo `ProveedorRUT` | `NroLicitacion`, `CodigoProductoONU`, `ProveedorRUT` | 2 |
| Monto atípico | Oferta fuera del rango IQR de su ítem y moneda | `CodigoProductoONU`, `MonedaOferta`, `MontoTotalOferta` | 3 |
| Brecha de adjudicación | Ganadora muy por encima de la oferta más baja del mismo ítem | `ResultadoOferta`, `MontoTotalOferta` | 3 |
| Plazo inusual | Días entre publicación y cierre en el extremo inferior de su tipo de licitación | `FechaPublicacion`, `FechaCierre`, `TipoLicitacion` | 4 |
| Concentración | Proporción de adjudicaciones por proveedor y por par proveedor–organismo | `ProveedorRUT`, `UnidadCompraRUT`, `Institucion` | 5 |
| Proveedores relacionados | RUT distintos con nombre, actividad o texto de oferta casi idénticos que coinciden en licitaciones u organismos | `Proveedor`, `ActividadProveedor`, `EspecificacionesProveedor`, `NombreOferta` | 6 |

Los umbrales definitivos (por ejemplo, el factor del IQR o el percentil de plazo) se fijan y justifican en la Fase 3.

## 2. Verificación del entorno

Se comprueba que el cuaderno se ejecuta con el intérprete del entorno virtual del proyecto y que las librerías están instaladas.

In [5]:
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "sklearn"]
NOMBRE_EN_PYPI = {"sklearn": "scikit-learn"}  # se instala scikit-learn, se importa sklearn


def verificar_entorno(dependencias):
    '''
    Comprueba intérprete, entorno virtual, carpeta de trabajo y librerías.

    Parámetros
    ----------
    dependencias : list of str
        Nombres de importación de las librerías que el proyecto necesita.

    Retorna
    -------
    dict
        Nombre de importación -> versión instalada (None si falta).
    '''
    en_venv = sys.prefix != sys.base_prefix
    print("Python              :", platform.python_version())
    print("Sistema operativo   :", platform.system(), platform.release())
    print("Intérprete          :", Path(sys.executable).as_posix())
    print("Entorno virtual     :", "[OK] activo" if en_venv else "[AVISO] se usa el intérprete del sistema")
    print("Carpeta de trabajo  :", Path.cwd().as_posix())

    versiones = {}
    print("\nLibrerías")
    for nombre in dependencias:
        paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
        try:
            versiones[nombre] = metadata.version(paquete)
            print(f"  [OK]    {paquete:<14} {versiones[nombre]}")
        except metadata.PackageNotFoundError:
            versiones[nombre] = None
            print(f"  [FALTA] {paquete}")
    return versiones


VERSIONES = verificar_entorno(DEPENDENCIAS)

Python              : 3.12.3
Sistema operativo   : Darwin 25.5.0
Intérprete          : /Users/oscar/Documents/Magister CS e IA/repositorio/proyecto-grupo2-mcdi500/.venv/bin/python
Entorno virtual     : [OK] activo
Carpeta de trabajo  : /Users/oscar/Documents/Magister CS e IA/repositorio/proyecto-grupo2-mcdi500/F1

Librerías
  [OK]    numpy          2.5.4
  [OK]    pandas         3.1.0
  [OK]    matplotlib     3.11.2
  [OK]    seaborn        0.13.2
  [OK]    scipy          1.18.1
  [OK]    scikit-learn   1.9.1


## 3. Estructura del repositorio y artefactos

Se localiza la raíz del proyecto y se comprueba que existan las carpetas y los archivos acordados. `.gitignore`, `requirements.txt` y los README ya están versionados, por lo que aquí solo se verifican.

In [6]:
NOMBRE_DATASET = "04LicitacionesEne.csv"


def localizar_raiz(inicio=None):
    '''
    Devuelve la raíz del proyecto subiendo desde la carpeta actual hasta
    encontrar una que contenga `data/raw` y `src`.

    Lanza
    -----
    FileNotFoundError
        Si no se encuentra una carpeta con esa estructura.
    '''
    base = Path(inicio or Path.cwd()).resolve()
    for carpeta in (base, *base.parents):
        if (carpeta / "data" / "raw").is_dir() and (carpeta / "src").is_dir():
            return carpeta
    raise FileNotFoundError(f"No se encontró la raíz del proyecto desde {base}.")


RAIZ = localizar_raiz()
DIR_CRUDO = RAIZ / "data" / "raw"
RUTA_DATASET = DIR_CRUDO / NOMBRE_DATASET
print("Raíz del proyecto:", RAIZ.as_posix())

Raíz del proyecto: /Users/oscar/Documents/Magister CS e IA/repositorio/proyecto-grupo2-mcdi500


In [7]:
CARPETAS_ESPERADAS = ["data/raw", "data/processed", "docs", "src", "F1", "F2", "F3", "F4"]
ARCHIVOS_ESPERADOS = [
    ".gitignore", "requirements.txt", "README.md", "docs/README.md",
    "src/utilidades.py", "src/validar_dataset.py", "validar_dataset.py",
    "F1/F1_Definicion.ipynb", "F2/F2_Preprocesamiento.ipynb",
    f"data/raw/{NOMBRE_DATASET}",
]


def verificar_estructura(raiz, carpetas, archivos):
    '''
    Comprueba que existan las carpetas y archivos comprometidos y que no estén vacíos.

    Retorna
    -------
    DataFrame
        Una fila por elemento con su estado.
    '''
    filas = [("carpeta", ruta, (raiz / ruta).is_dir()) for ruta in carpetas]
    for ruta in archivos:
        destino = raiz / ruta
        filas.append(("archivo", ruta, destino.is_file() and destino.stat().st_size > 0))
    return pd.DataFrame(filas, columns=["tipo", "ruta", "ok"])


estructura = verificar_estructura(RAIZ, CARPETAS_ESPERADAS, ARCHIVOS_ESPERADOS)
display(estructura.assign(estado=estructura["ok"].map({True: "OK", False: "FALTA"}))[["tipo", "ruta", "estado"]])

,tipo,ruta,estado
0,carpeta,data/raw,OK
1,carpeta,data/processed,OK
2,carpeta,docs,OK
3,carpeta,src,OK
4,carpeta,F1,OK
5,carpeta,F2,OK
6,carpeta,F3,OK
7,carpeta,F4,OK
8,archivo,.gitignore,OK
9,archivo,requirements.txt,OK


### 3.1 Dependencias y exclusiones

Se compara `requirements.txt` con las versiones instaladas y se revisa que `.gitignore` excluya el entorno virtual y las cachés.

In [8]:
def leer_requirements(ruta):
    '''Devuelve {paquete: version} a partir de las líneas `paquete==version`.'''
    pines = {}
    for linea in Path(ruta).read_text(encoding="utf-8").splitlines():
        linea = linea.strip()
        if linea and not linea.startswith("#") and "==" in linea:
            nombre, version = linea.split("==", 1)
            pines[nombre.strip().lower().replace("_", "-")] = version.strip()
    return pines


pines = leer_requirements(RAIZ / "requirements.txt")
gitignore = (RAIZ / ".gitignore").read_text(encoding="utf-8")

print(f"requirements.txt declara {len(pines)} paquetes con versión exacta.")
filas = []
for nombre in DEPENDENCIAS:
    paquete = NOMBRE_EN_PYPI.get(nombre, nombre).lower()
    filas.append({
        "paquete": paquete,
        "en_requirements": pines.get(paquete),
        "instalada": VERSIONES[nombre],
        "coincide": pines.get(paquete) == VERSIONES[nombre],
    })
display(pd.DataFrame(filas))

for patron in (".venv/", "__pycache__/", ".ipynb_checkpoints/"):
    print(f".gitignore excluye {patron:<20}:", patron in gitignore)
peso_mb = RUTA_DATASET.stat().st_size / 1024**2
print(f"\nTamaño del CSV original: {peso_mb:.1f} MB (límite de GitHub: 100 MB)")

requirements.txt declara 120 paquetes con versión exacta.


,paquete,en_requirements,instalada,coincide
0,numpy,2.5.4,2.5.4,True
1,pandas,3.1.0,3.1.0,True
2,matplotlib,3.11.2,3.11.2,True
3,seaborn,0.13.2,0.13.2,True
4,scipy,1.18.1,1.18.1,True
5,scikit-learn,1.9.1,1.9.1,True


.gitignore excluye .venv/              : False
.gitignore excluye __pycache__/        : True
.gitignore excluye .ipynb_checkpoints/ : False

Tamaño del CSV original: 58.7 MB (límite de GitHub: 100 MB)


## 4. Funciones reutilizables en `src/`

Las funciones de carga y conversión que usa la Fase 2 están en `src/utilidades.py`. Se importan y se prueban con tablas mínimas creadas en memoria (caso normal, límite y excepción), sin tocar el CSV original.

In [9]:
DIR_SRC = RAIZ / "src"
if str(DIR_SRC) not in sys.path:
    sys.path.insert(0, str(DIR_SRC))

import utilidades
importlib.reload(utilidades)

publicas = [n for n in dir(utilidades) if callable(getattr(utilidades, n)) and not n.startswith("_")
            and getattr(getattr(utilidades, n), "__module__", "") == "utilidades"]
print("Importado:", Path(utilidades.__file__).relative_to(RAIZ).as_posix())
print("Funciones del módulo:", ", ".join(publicas))

Importado: src/utilidades.py
Funciones del módulo: cargar_datos_crudos, cargar_datos_procesados, convertir_fechas, convertir_montos, detectar_atipicos_iqr, obtener_raiz_proyecto, ordenar_tamano_proveedor


In [10]:
# Caso normal: fechas y montos con coma decimal
muestra = pd.DataFrame({
    "FechaPublicacion": ["27-12-2024 11:07", "05-01-2025 09:30"],
    "MontoTotalOferta": ["32773109,24", "1500"],
})
resultado = utilidades.convertir_montos(utilidades.convertir_fechas(muestra, ["FechaPublicacion"]),
                                        ["MontoTotalOferta"])
assert pd.api.types.is_datetime64_any_dtype(resultado["FechaPublicacion"])
assert resultado["MontoTotalOferta"].iloc[0] == 32773109.24
print("[OK] Caso normal: fechas y montos se convierten correctamente")

# Caso límite: valores inválidos y columnas ausentes
sucio = pd.DataFrame({"FechaPublicacion": ["no es fecha", None], "MontoTotalOferta": ["abc", None]})
limpio = utilidades.convertir_montos(utilidades.convertir_fechas(sucio, ["FechaPublicacion", "NoExiste"]),
                                     ["MontoTotalOferta", "NoExiste"])
assert limpio["FechaPublicacion"].isna().all() and limpio["MontoTotalOferta"].isna().all()
assert "NoExiste" not in limpio.columns
print("[OK] Caso límite: valores inválidos pasan a nulo y las columnas ausentes se ignoran")

# Caso excepción: archivo inexistente
try:
    utilidades.cargar_datos_crudos("archivo_inexistente.csv")
except FileNotFoundError as error:
    print("[OK] Caso excepción: FileNotFoundError ->", str(error)[:60], "...")
else:
    raise AssertionError("Se esperaba FileNotFoundError")

# La entrada no se modifica
assert muestra["MontoTotalOferta"].iloc[0] == "32773109,24"
print("[OK] Las funciones no modifican el DataFrame de entrada")

[OK] Caso normal: fechas y montos se convierten correctamente
[OK] Caso límite: valores inválidos pasan a nulo y las columnas ausentes se ignoran
[OK] Caso excepción: FileNotFoundError -> No se encontró el archivo original en: /Users/oscar/Document ...
[OK] Las funciones no modifican el DataFrame de entrada


## 5. Conjunto de datos

### 5.1 Procedencia

| Campo | Valor |
| --- | --- |
| Título | Licitaciones — enero 2025 (`04LicitacionesEne.csv`) |
| Autor | ChileCompra (Dirección de Compras y Contratación Pública) |
| Plataforma | Datos Abiertos ChileCompra |
| Enlace | `https://datos-abiertos.chilecompra.cl/descargas/ordenes-y-licitaciones` |
| Estructura documentada | 45.741 filas × 74 columnas |
| Unidad de observación | Una oferta de un proveedor para un ítem de una licitación |
| Formato | CSV, separador `;`, codificación `latin-1` |

**Referencia (APA 7):**

> ChileCompra. (s. f.). *Datos abiertos* [Conjunto de datos]. https://datos-abiertos.chilecompra.cl/descargas/ordenes-y-licitaciones

*Pendiente: completar la fecha de descarga y el nombre exacto de la descarga usada.*

### 5.2 Evaluación contra los criterios del curso

Se usan las funciones `perfilar` y `evaluar` del validador compartido (`src/validar_dataset.py`) sobre el archivo real.

In [11]:
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.validar_dataset import perfilar, evaluar

if not RUTA_DATASET.is_file():
    raise FileNotFoundError(f"No se encontró el archivo original: {RUTA_DATASET}")

MTIME_INICIAL = RUTA_DATASET.stat().st_mtime
df_crudo = pd.read_csv(RUTA_DATASET, sep=";", encoding="latin-1", low_memory=False)
perfil = perfilar(df_crudo)
chequeo, apto = evaluar(df_crudo, perfil, RUTA_DATASET)

print("Archivo original :", RUTA_DATASET.relative_to(RAIZ).as_posix())
print("Dimensiones      :", df_crudo.shape)
display(chequeo)

Archivo original : data/raw/04LicitacionesEne.csv
Dimensiones      : (45741, 74)


,estado,requisito,detalle
0,OK,Al menos 2000 filas,45741 filas
1,OK,Al menos 12 columnas,74 columnas
2,OK,Combina al menos 3 roles analíticos,"alta cardinalidad, nominal, fecha, binaria, di..."
3,OK,Al menos una variable numérica,7 numéricas (continuas o discretas)
4,OK,Al menos una variable categórica,"26 categóricas (nominales, binarias u ordinales)"
5,OK,Presencia de valores faltantes,máximo 100.0% en una variable
6,AVISO,Ninguna variable sobre 60% de faltantes,máximo 100.0%
7,OK,Incluye alguna variable de fecha,12 detectadas
8,OK,Incluye texto o categórica de alta cardinalidad,26 detectadas
9,OK,Archivo bajo 100 MB,58.7 MB


In [12]:
faltan = chequeo.loc[chequeo["estado"] == "FALTA", "requisito"].tolist()
avisos = chequeo.loc[chequeo["estado"] == "AVISO", "requisito"].tolist()
print(f"Criterios evaluados     : {len(chequeo)}")
print(f"Obligatorios sin cumplir: {len(faltan)}")
print(f"Avisos                  : {len(avisos)}")
for texto in avisos:
    print(f"  · {texto}")
print("\nRESULTADO:", "el conjunto CUMPLE los requisitos obligatorios." if apto else "el conjunto NO está aprobado.")

print("\nComposición por rol analítico (según el validador):")
display(perfil["rol"].value_counts().to_frame("variables"))

Criterios evaluados     : 11
Obligatorios sin cumplir: 0
Avisos                  : 1
  · Ninguna variable sobre 60% de faltantes

RESULTADO: el conjunto CUMPLE los requisitos obligatorios.

Composición por rol analítico (según el validador):


,variables
rol,
alta cardinalidad,20
nominal,15
fecha,12
binaria,11
discreta,7
texto libre,6
vacía,3


El único aviso corresponde a las tres columnas totalmente vacías (`LicitacionBaseTipo`, `ContratoRenovable`, `UnidadTiempoRenovacion`), que se tratan en la Fase 2. El validador clasifica `MontoNetoOferta` y `MontoTotalOferta` como «alta cardinalidad» porque el archivo trae los montos como texto con coma decimal; su rol real es numérico continuo, y por eso el diccionario siguiente declara el rol esperado junto al detectado.

### 5.3 Variables candidatas y su rol analítico

El diccionario completo de las 74 columnas está en `docs/README.md`. Estas son las variables que sostienen los objetivos:

| Uso | Variables | Objetivos |
| --- | --- | --- |
| Proceso e ítem | `NroLicitacion`, `CodigoProductoONU`, `NombreItem`, `TipoLicitacion` | 1–4 |
| Comprador | `UnidadCompraRUT`, `UnidadCompra`, `Institucion` | 1, 5, 6 |
| Proveedor | `ProveedorRUT`, `Proveedor`, `ActividadProveedor`, `TamanoProveedor` | 2, 5, 6 |
| Contenido de la oferta | `NombreOferta`, `EspecificacionesProveedor` | 6 |
| Rubro | `RubroN1`, `RubroN2` | 1, 2 |
| Tiempo | `FechaPublicacion`, `FechaCierre`, `FechaAdjudicacion` | 4 |
| Resultado y monto | `ResultadoOferta`, `MonedaOferta`, `MontoTotalOferta` | 1, 3, 5 |

In [13]:
# (variable, rol analítico esperado, comentario)
DICCIONARIO = [
    ("NroLicitacion",             "identificador",     "Identifica la licitación; se repite en varias ofertas"),
    ("TipoLicitacion",            "nominal",           "Modalidad (L1, LE, LP...); referencia para plazos"),
    ("CodigoProductoONU",         "nominal",           "Código del ítem: define ofertas comparables"),
    ("NombreItem",                "alta cardinalidad", "Nombre del ítem licitado"),
    ("UnidadCompraRUT",           "identificador",     "Unidad compradora"),
    ("UnidadCompra",              "alta cardinalidad", "Nombre de la unidad compradora"),
    ("Institucion",               "alta cardinalidad", "Organismo comprador"),
    ("ProveedorRUT",              "identificador",     "Proveedor oferente"),
    ("Proveedor",                 "alta cardinalidad", "Razón social: base para similitud de nombres"),
    ("ActividadProveedor",        "texto libre",       "Giro declarado: base para similitud de actividad"),
    ("TamanoProveedor",           "ordinal",           "Micro, Pequeña, Mediana, Grande"),
    ("NombreOferta",              "alta cardinalidad", "Título de la oferta"),
    ("EspecificacionesProveedor", "texto libre",       "Texto de la oferta: base para detectar textos idénticos"),
    ("RubroN1",                   "alta cardinalidad", "Rubro de primer nivel"),
    ("RubroN2",                   "alta cardinalidad", "Rubro de segundo nivel"),
    ("FechaPublicacion",          "fecha",             "Inicio del plazo; llega como texto"),
    ("FechaCierre",               "fecha",             "Fin del plazo; llega como texto"),
    ("FechaAdjudicacion",         "fecha",             "Puede venir vacía"),
    ("ResultadoOferta",           "binaria",           "Ganadora / Perdedora"),
    ("MonedaOferta",              "nominal",           "CLP, CLF, USD, UTM: no mezclar"),
    ("MontoTotalOferta",          "continua",          "Llega como texto con coma decimal"),
]
diccionario = pd.DataFrame(DICCIONARIO, columns=["variable", "rol_esperado", "comentario"])
diccionario["rol_detectado"] = diccionario["variable"].map(perfil.set_index("variable")["rol"])
diccionario["pct_nulos"] = diccionario["variable"].map(perfil.set_index("variable")["pct_nulos"])
display(diccionario)

,variable,rol_esperado,comentario,rol_detectado,pct_nulos
0,NroLicitacion,identificador,Identifica la licitación; se repite en varias ...,alta cardinalidad,0.0
1,TipoLicitacion,nominal,"Modalidad (L1, LE, LP...); referencia para plazos",nominal,0.0
2,CodigoProductoONU,nominal,Código del ítem: define ofertas comparables,discreta,0.0
3,NombreItem,alta cardinalidad,Nombre del ítem licitado,alta cardinalidad,0.0
4,UnidadCompraRUT,identificador,Unidad compradora,alta cardinalidad,0.0
5,UnidadCompra,alta cardinalidad,Nombre de la unidad compradora,alta cardinalidad,0.0
6,Institucion,alta cardinalidad,Organismo comprador,alta cardinalidad,0.0
7,ProveedorRUT,identificador,Proveedor oferente,alta cardinalidad,0.0
8,Proveedor,alta cardinalidad,Razón social: base para similitud de nombres,alta cardinalidad,0.0
9,ActividadProveedor,texto libre,Giro declarado: base para similitud de actividad,texto libre,22.2


### 5.4 Viabilidad de las señales de alerta

Antes de comprometer los objetivos se comprueba que cada señal puede calcularse con los datos. Son conteos sobre el archivo en memoria: no se limpia ni se guarda nada, y las cifras son preliminares.

In [14]:
CLAVE_ITEM = ["NroLicitacion", "CodigoProductoONU", "NombreItem"]

print("Resultado de la oferta:")
display(df_crudo["ResultadoOferta"].value_counts(dropna=False).to_frame("ofertas"))
print("Ofertas por moneda:")
display(df_crudo["MonedaOferta"].value_counts(dropna=False).to_frame("ofertas"))
print(f"Filas: {len(df_crudo):,} · licitaciones distintas: {df_crudo['NroLicitacion'].nunique():,} "
      f"· ítems distintos: {df_crudo.groupby(CLAVE_ITEM).ngroups:,}")

Resultado de la oferta:


,ofertas
ResultadoOferta,
Perdedora,25853
Ganadora,19888


Ofertas por moneda:


,ofertas
MonedaOferta,
CLP,45604
CLF,105
USD,23
UTM,9


Filas: 45,741 · licitaciones distintas: 2,292 · ítems distintos: 4,707


In [15]:
# Objetivo 2 — baja competencia
oferentes_item = df_crudo.groupby(CLAVE_ITEM)["ProveedorRUT"].nunique()
oferentes_lic = df_crudo.groupby("NroLicitacion")["ProveedorRUT"].nunique()
print("Baja competencia")
print(f"  Ítems con un solo oferente        : {(oferentes_item == 1).sum():,} de {len(oferentes_item):,} "
      f"({(oferentes_item == 1).mean():.1%})")
print(f"  Licitaciones con un solo oferente : {(oferentes_lic == 1).sum():,} de {len(oferentes_lic):,} "
      f"({(oferentes_lic == 1).mean():.1%})")

# Objetivo 4 — plazos (conversión solo en memoria, para medir viabilidad)
fecha = lambda col: pd.to_datetime(df_crudo[col], format="%d-%m-%Y %H:%M", errors="coerce")
dias = (fecha("FechaCierre") - fecha("FechaPublicacion")).dt.total_seconds() / 86400
plazos = (df_crudo.assign(dias=dias).drop_duplicates("NroLicitacion")
          .groupby("TipoLicitacion")["dias"].describe()[["count", "min", "25%", "50%"]].round(1))
print("\nDías entre publicación y cierre, por tipo de licitación:")
display(plazos)

Baja competencia
  Ítems con un solo oferente        : 751 de 4,707 (16.0%)
  Licitaciones con un solo oferente : 555 de 2,292 (24.2%)

Días entre publicación y cierre, por tipo de licitación:


,count,min,25%,50%
TipoLicitacion,,,,
Licitación Privada Mayor a 1000 UTM,4.0,10.9,11.1,15.6
Licitación Privada Menor a 100 UTM.,3.0,6.0,6.8,7.7
Licitación Privada entre 100 y 1000 UTM.,16.0,5.0,6.8,8.0
Licitación Pública Entre 100 y 1000 UTM (LE),1453.0,5.0,6.9,9.8
Licitación Pública Mayor 1000 UTM (LP),178.0,10.0,12.3,19.8
Licitación Pública Mayor a 5000 (LR),79.0,30.0,30.9,31.9
Licitación Pública Menor a 100 UTM (L1),396.0,4.6,6.1,6.9
Licitación Pública Servicios personales especializados (LS),1.0,32.8,32.8,32.8
Licitación Pública entre a 2000 y 5000 UTM (LQ),162.0,10.0,19.3,20.2


In [16]:
# Objetivo 5 — concentración de adjudicaciones
ganadoras = df_crudo[df_crudo["ResultadoOferta"] == "Ganadora"]
por_proveedor = ganadoras["ProveedorRUT"].value_counts()
print("Concentración")
print(f"  Ofertas ganadoras                 : {len(ganadoras):,}")
print(f"  Proveedores ganadores distintos   : {len(por_proveedor):,}")
print(f"  Participación de los 10 primeros  : {por_proveedor.head(10).sum() / len(ganadoras):.1%}")
pares = ganadoras.groupby(["ProveedorRUT", "UnidadCompraRUT"]).size()
print(f"  Pares proveedor–comprador con ≥ 5 adjudicaciones: {(pares >= 5).sum():,}")

# Objetivo 6 — posibles proveedores relacionados
nombre_norm = df_crudo["Proveedor"].str.strip().str.upper()
rut_por_nombre = df_crudo.assign(nombre=nombre_norm).groupby("nombre")["ProveedorRUT"].nunique()
textos = df_crudo.dropna(subset=["EspecificacionesProveedor"])
textos = textos[textos["EspecificacionesProveedor"].str.len() > 40]
rut_por_texto = textos.groupby(["NroLicitacion", "EspecificacionesProveedor"])["ProveedorRUT"].nunique()
print("\nPosibles proveedores relacionados")
print(f"  Nombres idénticos con más de un RUT                    : {(rut_por_nombre > 1).sum()} "
      f"({', '.join(rut_por_nombre[rut_por_nombre > 1].index)})")
print(f"  Textos de oferta idénticos de RUT distintos en una misma licitación: {(rut_por_texto > 1).sum():,}")
print(f"  Actividades declaradas distintas                       : {df_crudo['ActividadProveedor'].nunique():,}")

Concentración
  Ofertas ganadoras                 : 19,888
  Proveedores ganadores distintos   : 2,141
  Participación de los 10 primeros  : 21.7%
  Pares proveedor–comprador con ≥ 5 adjudicaciones: 696

Posibles proveedores relacionados
  Nombres idénticos con más de un RUT                    : 2 (PERSONA NATURAL, SUR SPA)
  Textos de oferta idénticos de RUT distintos en una misma licitación: 564
  Actividades declaradas distintas                       : 2,374


**Lectura preliminar.**

- **Competencia, plazos y concentración** pueden calcularse directamente: hay suficientes ítems con un solo oferente, y los plazos varían de forma clara según el tipo de licitación.
- **Montos y brecha de adjudicación** requieren convertir los montos a número (Fase 2) y comparar solo dentro del mismo ítem y moneda.
- **Proveedores relacionados:** la coincidencia exacta de nombres casi no aparece y los casos encontrados son genéricos o anonimizados («PERSONA NATURAL»). En cambio, los textos de oferta idénticos presentados por RUT distintos en una misma licitación son frecuentes. Esto sugiere medir la **similitud aproximada** de nombres, giro y texto de las ofertas, y filtrar los textos genéricos (por ejemplo, «según bases»), en lugar de buscar coincidencias exactas. Como el archivo no trae socios ni domicilios, el resultado será una lista de pares para revisión, no una prueba de vínculo.

## 6. Control de versiones

Estado del repositorio al ejecutar el cuaderno. No se hacen `add` ni `commit` desde aquí.

In [17]:
def git(*argumentos):
    '''
    Ejecuta un comando de Git en la raíz del proyecto.

    Retorna
    -------
    (bool, str)
        Éxito de la ejecución y salida (o mensaje de error).
    '''
    if shutil.which("git") is None:
        return False, "Git no está instalado o no está en el PATH."
    try:
        proceso = subprocess.run(["git", *argumentos], cwd=RAIZ, capture_output=True,
                                 text=True, check=False)
    except OSError as error:
        return False, str(error)
    return proceso.returncode == 0, (proceso.stdout or proceso.stderr).strip()


es_repo, _ = git("rev-parse", "--is-inside-work-tree")
print("Carpeta bajo Git :", es_repo)
if es_repo:
    print("Rama actual      :", git("rev-parse", "--abbrev-ref", "HEAD")[1])
    print("Usuario de Git   :", git("config", "user.name")[1] or "(sin configurar)")
    print("\nÚltimos commits")
    print(git("log", "--oneline", "-5")[1] or "  (todavía no hay commits)")

    ok, autores = git("shortlog", "-sn", "--all", "--no-merges")
    print("\nCommits por autor (sin merges)")
    print(autores if ok and autores else "  (sin datos)")

    ok, cambios = git("status", "--porcelain")
    print("\nCambios sin confirmar:", len(cambios.splitlines()) if cambios else 0)

Carpeta bajo Git : True
Rama actual      : feat-oscar


Usuario de Git   : Oscar-Salgado

Últimos commits


27fc574 chore: actualizar estructura del repositorio en README.md [skip ci]
ffc8f60 Agregar notebook de limpieza de F2
e2476e0 chore: actualizar estructura del repositorio en README.md [skip ci]
cca9ae0 Merge pull request #16 from D-Rojas-S/utilidades
e406c3f se crea documento utilidades.py



Commits por autor (sin merges)
8	Hiigarian
     8	Oscar-Salgado
     4	Bert Gonzalez Olmos
     4	Bertgil Gonzalez Olmos
     3	github-actions[bot]



Cambios sin confirmar: 4


## 7. Validación de la fase

`validar_fase1` reúne las comprobaciones anteriores y detiene el cuaderno si alguna falla: estructura, dependencias, definición del proyecto, criterios del conjunto, variables candidatas, módulo importable y archivo original intacto.

In [18]:
def validar_fase1(raiz, config, estructura, pines, versiones, apto, dicc, columnas_csv, mtime_inicial):
    '''
    Comprueba que la Fase 1 dejó lo comprometido.
    Lanza AssertionError si alguna comprobación falla.
    '''
    print("VALIDACIÓN DE LA FASE 1")
    print("-" * 45)

    faltan = estructura.loc[~estructura["ok"], "ruta"].tolist()
    assert not faltan, f"Faltan elementos del repositorio: {faltan}"
    print(f"[OK] Estructura: {len(estructura)} elementos presentes")

    assert pines, "requirements.txt no fija ninguna versión"
    desajuste = [n for n, v in versiones.items()
                 if v is None or pines.get(NOMBRE_EN_PYPI.get(n, n).lower()) != v]
    assert not desajuste, f"requirements.txt no coincide con el entorno en: {desajuste}"
    print(f"[OK] Dependencias: {len(pines)} fijadas y coherentes con el entorno")

    faltan_claves = [c for c in CLAVES_OBLIGATORIAS if not config.get(c)]
    assert not faltan_claves, f"Definición incompleta: {faltan_claves}"
    assert len(config["objetivos_especificos"]) == len(config["preguntas"])
    print(f"[OK] Definición: {len(config['objetivos_especificos'])} objetivos con su pregunta asociada")

    assert apto, "El conjunto no cumple los criterios obligatorios del curso"
    ausentes = sorted(set(dicc["variable"]) - set(columnas_csv))
    assert not ausentes, f"Variables candidatas ausentes del archivo: {ausentes}"
    print(f"[OK] Datos: criterios cumplidos y {len(dicc)} variables candidatas presentes")

    assert hasattr(utilidades, "cargar_datos_crudos") and hasattr(utilidades, "convertir_montos")
    print("[OK] Módulo: src/utilidades.py importable")

    assert (raiz / "data" / "raw" / NOMBRE_DATASET).stat().st_mtime == mtime_inicial
    print("[OK] El archivo original no fue modificado")
    print("-" * 45)
    print("Fase 1 validada.")


validar_fase1(RAIZ, PROYECTO, estructura, pines, VERSIONES, apto, diccionario,
              df_crudo.columns, MTIME_INICIAL)

VALIDACIÓN DE LA FASE 1
---------------------------------------------
[OK] Estructura: 18 elementos presentes
[OK] Dependencias: 120 fijadas y coherentes con el entorno
[OK] Definición: 6 objetivos con su pregunta asociada
[OK] Datos: criterios cumplidos y 21 variables candidatas presentes
[OK] Módulo: src/utilidades.py importable
[OK] El archivo original no fue modificado
---------------------------------------------
Fase 1 validada.


## 8. Vinculación con el mapa conceptual

*Pendiente del grupo: ajustar los nombres de los nodos para que coincidan con el mapa conceptual entregado en la Formativa 1.*

In [19]:
VINCULACION = [
    # (nodo del mapa conceptual, dónde se implementa, evidencia)
    ("Problema y objetivos",      "Sección 1 · PROYECTO",               "Objetivos, preguntas y señales de alerta"),
    ("Datos y fuente",            "Sección 5 · data/raw/",              "Ficha, validador y perfil del CSV original"),
    ("Entorno y dependencias",    "Secciones 2 y 3.1",                  "Intérprete, versiones y requirements.txt"),
    ("Estructura del proyecto",   "Sección 3 · carpetas F1–F4",         "Tabla de verificación de estructura"),
    ("Funciones reutilizables",   "Sección 4 · src/utilidades.py",      "Pruebas de caso normal, límite y excepción"),
    ("Limpieza y preparación",    "F2/F2_Preprocesamiento.ipynb",       "Conversión de fechas y montos"),
    ("Indicadores de alerta",     "F3",                                 "Competencia, montos, plazos, concentración y similitud"),
    ("Comunicación de resultados","F4",                                 "Informe final y visualizaciones"),
    ("Control de versiones",      "Sección 6 · git",                    "Ramas, autores y commits"),
]
vinculacion = pd.DataFrame(VINCULACION, columns=["nodo_conceptual", "implementacion", "evidencia"])
display(vinculacion)

,nodo_conceptual,implementacion,evidencia
0,Problema y objetivos,Sección 1 · PROYECTO,"Objetivos, preguntas y señales de alerta"
1,Datos y fuente,Sección 5 · data/raw/,"Ficha, validador y perfil del CSV original"
2,Entorno y dependencias,Secciones 2 y 3.1,"Intérprete, versiones y requirements.txt"
3,Estructura del proyecto,Sección 3 · carpetas F1–F4,Tabla de verificación de estructura
4,Funciones reutilizables,Sección 4 · src/utilidades.py,"Pruebas de caso normal, límite y excepción"
5,Limpieza y preparación,F2/F2_Preprocesamiento.ipynb,Conversión de fechas y montos
6,Indicadores de alerta,F3,"Competencia, montos, plazos, concentración y s..."
7,Comunicación de resultados,F4,Informe final y visualizaciones
8,Control de versiones,Sección 6 · git,"Ramas, autores y commits"


### Trazabilidad de las preguntas

| Pregunta | Variables principales | Fase | Evidencia prevista |
| --- | --- | --- | --- |
| 1. Distribución de ofertas y resultados | `ResultadoOferta`, `TipoLicitacion`, `RubroN1`, `Institucion` | F3 | Tablas de frecuencia y proporciones |
| 2. Baja competencia | `NroLicitacion`, `CodigoProductoONU`, `ProveedorRUT` | F3 | Proporción de ítems y licitaciones con un oferente |
| 3. Montos atípicos y brecha de adjudicación | `CodigoProductoONU`, `MonedaOferta`, `MontoTotalOferta`, `ResultadoOferta` | F2 y F3 | Rango IQR por ítem y moneda; brecha ganadora–mínima |
| 4. Plazos inusuales | `FechaPublicacion`, `FechaCierre`, `TipoLicitacion` | F2 y F3 | Distribución de días por tipo y casos extremos |
| 5. Concentración y patrones repetitivos | `ProveedorRUT`, `UnidadCompraRUT`, `Institucion` | F3 | Participación acumulada y pares proveedor–comprador |
| 6. Proveedores relacionados | `Proveedor`, `ActividadProveedor`, `EspecificacionesProveedor`, `NroLicitacion` | F3 y F4 | Pares de RUT con alta similitud que coinciden en licitaciones u organismos |

## 9. Decisiones por cerrar

- **Oferta ganadora:** `ResultadoOferta` presenta `Ganadora` y `Perdedora`; cada tasa debe declarar su denominador (ofertas, ítems, licitaciones o proveedores).
- **Ítems comparables:** definir si la comparación de montos se hace por `CodigoProductoONU`, por `NombreItem` o por ítem dentro de la licitación, y cómo tratar las diferencias de cantidad (`CantidadOferta`).
- **Monedas:** analizar por moneda o definir una conversión con fecha de referencia.
- **Plazos:** fijar el umbral de «plazo inusual» (percentil dentro del tipo de licitación) y contrastarlo con los plazos mínimos de la normativa vigente.
- **Similitud de proveedores:** elegir la medida de similitud de texto, el umbral y la lista de textos genéricos que se excluyen; decidir si se incorporan fuentes externas para confirmar vínculos.
- **Geografía:** el archivo no tiene una variable geográfica explícita; el objetivo geográfico del `README` debe ajustarse.

## Conclusiones

La fase deja definido el problema con seis objetivos específicos, cada uno con su pregunta y su señal de alerta; un entorno verificado; una estructura de repositorio con sus artefactos de reproducibilidad; un módulo propio importable y probado; y el conjunto de datos documentado, evaluado contra los criterios del curso y revisado en cuanto a la viabilidad de cada indicador.

## 10. Reflexión técnica

> *Borrador para revisión del grupo. Las cifras provienen de la ejecución de este cuaderno.*

### Hallazgos

- **El conjunto cumple los requisitos obligatorios** (45.741 filas × 74 columnas, con variables numéricas, categóricas, fechas y texto). El único aviso son las columnas totalmente vacías.
- **La unidad de análisis cambia las cifras.** Las filas son ofertas por ítem, no licitaciones; cualquier proporción debe declarar su denominador.
- **Las señales de alerta son viables con los datos disponibles.** Una parte relevante de los ítems recibe una sola oferta, los plazos difieren según el tipo de licitación y hay textos de oferta idénticos presentados por RUT distintos en una misma licitación.
- **La búsqueda de proveedores relacionados no puede basarse en coincidencias exactas.** Los nombres idénticos con distinto RUT son casi inexistentes y corresponden a nombres genéricos; la señal útil está en la similitud aproximada de nombres, giro y texto de las ofertas.
- **El rol analítico difiere del tipo de dato.** Montos y fechas llegan como texto; el diccionario deja anticipadas sus conversiones en la Fase 2.

### Dificultades

- La similitud entre proveedores es solo un indicio: sin socios ni domicilios, no se puede afirmar que dos RUT pertenezcan al mismo grupo.
- Varias señales (un oferente, plazo corto) pueden tener explicaciones legítimas, por lo que deben interpretarse junto con el rubro y el tipo de licitación.

### Decisiones abiertas

Ver la sección 9.

## 11. Lista de entrega

- [ ] El cuaderno corre completo con *Restart Kernel and Run All Cells*, sin errores.
- [ ] Problemática, objetivos, preguntas, alcance y limitaciones declarados.
- [ ] Procedencia documentada con cita APA 7 y fecha de descarga.
- [ ] Variables candidatas con su rol analítico.
- [ ] Conjunto evaluado contra los criterios del curso.
- [ ] Pruebas de casos normal, límite y excepción.
- [ ] Nodos de la vinculación iguales a los del mapa conceptual.
- [ ] *Commits* de todos los integrantes, con el correo de Git igual al de GitHub.
- [ ] `README` actualizado con el nombre `F1/F1_Definicion.ipynb` y los objetivos de este cuaderno.